In [1]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(".."))
pd.set_option("display.max_columns", None)

from src.services.transformer import FeatureTransformer
from src.services.train import Trainer

In [2]:
df = pd.read_csv("../data/engineered_data.csv")
df.head()

,store,dayofweek,promo,stateholiday,schoolholiday,sales,month,day,week_of_year
0,1,5,1,0,1,5263,7,31,31
1,2,5,1,0,1,6064,7,31,31
2,3,5,1,0,1,8314,7,31,31
3,4,5,1,0,1,13995,7,31,31
4,5,5,1,0,1,4822,7,31,31


In [3]:
size = int(df.shape[0]*0.8)
train = df.iloc[:size]
test = df.iloc[size:]

In [4]:
feature_trans = FeatureTransformer()
feature_trans.fit(train.copy())
train_trans = feature_trans.transform(train)
test_trans = feature_trans.transform(test)

In [5]:
X_train = train_trans
y_train = train["sales"]

In [6]:
X_test = test_trans
y_test = test["sales"]

In [7]:
trainer = Trainer()
model, result = trainer.train(X_train, y_train, X_test, y_test)

model: decision_tree
decision_tree MAE: 2021.458303326952
model: random_forest
random_forest MAE: 2000.404652754717

Best Model: RandomForestRegressor(max_depth=10, min_samples_leaf=5, min_samples_split=10,
                      n_estimators=200, n_jobs=-1, random_state=42)
Best MAE: 2000.404652754717


In [8]:
train_prediction = model.predict(X_train)
test_prediction = model.predict(X_test)

train_mae = mean_absolute_error(y_train, train_prediction)
test_mae = mean_absolute_error(y_test, test_prediction)

print(f"train mae: {train_mae}")
print(f"test mae : {test_mae}")

train mae: 1903.3798323786539
test mae : 2000.404652754717


### test training pipeline

In [9]:
from src.pipelines.training_pipeline import TrainingPipeline

In [10]:
tp = TrainingPipeline()

tp.run_training_pipeline()

shape rossmann: (1017209, 9)
shape store: (1115, 10)
shape after merge: (1017209, 18)
shape with test cols only: (1017209, 8)
converting columns to lowercase...
shape: (1017209, 8)
shape: (844392, 8)
shape: (844392, 7)
shape: (844392, 7)
shape: (844392, 9)
model: decision_tree
decision_tree MAE: 2021.458303326952
model: random_forest
random_forest MAE: 2000.404652754717

Best Model: RandomForestRegressor(max_depth=10, min_samples_leaf=5, min_samples_split=10,
                      n_estimators=200, n_jobs=-1, random_state=42)
Best MAE: 2000.404652754717
train mae: 1903.3798323786539
test mae : 2000.404652754717


(RandomForestRegressor(max_depth=10, min_samples_leaf=5, min_samples_split=10,
                       n_estimators=200, n_jobs=-1, random_state=42),
 <src.services.transformer.FeatureTransformer at 0x70d8d2606c90>)

### test prediction pipeline

In [11]:
from datetime import datetime
from pydantic import BaseModel
from src.pipelines.prediction_pipeline import PredictionPipeline

In [12]:
class PredictionInput(BaseModel):
    store: int
    day_of_week: int
    promo: int
    state_holiday:int
    school_holiday: int
    date: datetime

data = PredictionInput(
    store= 101,
    day_of_week=31,
    promo=1,
    state_holiday=1,
    school_holiday=1,
    date="2026-01-01"
)

input_df = pd.DataFrame([{
        "store": data.store,
        "dayofweek": data.day_of_week,
        "promo": data.promo,
        "stateholiday": data.state_holiday,
        "schoolholiday": data.school_holiday,
        "date": data.date
    }])

In [13]:
pp = PredictionPipeline()

In [14]:
pp.run_prediction_pipeline(input_df.copy())

shape: (1, 6)
shape: (1, 8)


array([10745.26994138])